### IMPORTAÇÕES

In [1]:
# IMPORTANDO BIBLIOTECAS
import pandas as pd
import awswrangler as awr
import openpyxl
import os
import shutil
import datetime as dt
import pyautogui
import time
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  

In [2]:
# FORMATANDO EM CAIXA ALTA
def format_type(df):
    for col in df.select_dtypes(include=['string']).columns:
        df[col] = df[col].str.upper()
    return df

### EXTRAÇÕES DE BASES

In [3]:
# EXTRAINDO DADOS DE ATIVOS

query_ativos = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_ATIVOS.sql"
with open(query_ativos, 'r', encoding='utf-8') as file:
    sql_ativos = file.read()   

df_ativos = awr.athena.read_sql_query(
    sql=sql_ativos,
    database='silver',
    ctas_approach=False
)



In [4]:
# ELIMINANDO DUPLICATAS DE ATIVOS POR CHASSI

df_ativos = df_ativos.drop(columns=['rn'])

df_ativos = (
    df_ativos
    .drop_duplicates(subset=['chassi'], keep='first')
)

df_ativos = df_ativos.sort_values(
    by=['inicio_vig', 'data_ativacao'], ascending=False
).reset_index(drop=True)

In [5]:
# EXTRAINDO DADOS DE CANCELAMENTOS INTEGRAIS (CONJUNTO)

query_cancelados_integrais = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_CANCELAMENTOS_INTEGRAIS.sql"
with open(query_cancelados_integrais, 'r', encoding='utf-8') as file:
    sql_cancelados_integrais = file.read()   

df_cancelamentos_integrais =awr.athena.read_sql_query(
    sql=sql_cancelados_integrais, database='silver',
    ctas_approach=False
)

In [6]:
# EXTRAINDO DADOS DE CANCELAMENTOS PARCIAIS (CONJUNTO)

query_cancelados_parciais = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\sql\all_boards_CANCELAMENTOS_PARCIAIS.sql"
with open(query_cancelados_parciais, 'r', encoding='utf-8') as file:
    sql_cancelados_parciais = file.read()   

df_cancelamentos_parciais =awr.athena.read_sql_query(
    sql=sql_cancelados_parciais,database='silver',
    ctas_approach=False
)

In [7]:
# FORMATAÇÃO EM CAIXA ALTA
format_type(df_ativos)
format_type(df_cancelamentos_integrais)
format_type(df_cancelamentos_parciais)

,placa,chassi,id_placa,id_veiculo,id_carroceria,matricula,conjunto,unidade,consultor,status,...,usuario_cancelamento,coverage_id,beneficio,status_beneficio,data_extracao,data_registro,data_ativacao,data_ativacao_beneficio,data_atualizacao,empresa
0,MLD9B48,9EP071120D1001811,9357,0,9357,17284,16550,UNIDADE PORTO VELHO,ELIZETE CHAGAS,ATIVO,...,<NA>,102290,RASTREADOR REBOQUE/SEMIRREBOQUE,CANCELADO,2026-05-13,2025-12-10,2025-12-30,2025-12-30,2026-05-05,STCOOP
1,MLD9B48,9EP071120D1001811,9357,0,9357,17284,16550,UNIDADE PORTO VELHO,ELIZETE CHAGAS,ATIVO,...,<NA>,102289,CASCO (R/SR),CANCELADO,2026-05-13,2025-12-10,2025-12-30,2025-12-30,2026-05-05,STCOOP
2,ACM7658,9ADG12430NC092508,18370,0,18370,19401,15349,UNIDADE CURITIBA,AMILTON MARTUCCI,ATIVO,...,<NA>,95301,RASTREADOR REBOQUE/SEMIRREBOQUE,CANCELADO,2026-05-13,2025-08-01,2025-08-03,2025-08-03,2026-02-04,STCOOP
3,ACM7658,9ADG12430NC092508,18370,0,18370,19401,15349,UNIDADE CURITIBA,AMILTON MARTUCCI,ATIVO,...,<NA>,95300,CASCO (R/SR),CANCELADO,2026-05-13,2025-08-01,2025-08-03,2025-08-03,2026-02-04,STCOOP
4,MLA3574,9BSR6X200D3834996,24230,24230,<NA>,20764,14317,UNIDADE ITAJAI,JOSÉ ELIAS DOS SANTOS FILHO,ATIVO,...,<NA>,88952,RASTREADOR,CANCELADO,2026-05-13,2025-05-02,2025-05-07,2025-05-07,2026-03-27,STCOOP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2099,BLT4G14,9A9RRTCA2P1BE5155,36792,0,36792,6718,10474,UNIDADE VILHENA,VILMA GIRIOLI,ATIVO,...,<NA>,783379,CASCO PT (R/SR),CANCELADO,2026-05-13,2026-02-05,2026-02-10,2026-02-10,2026-05-07,TAG
2100,FRF4348,93ZM2SSH0E8826134,60020,60020,<NA>,23984,140570,UNIDADE MARINGÁ,ICARO GOMES,ATIVO,...,<NA>,677634,ASSISTÊNCIA 24 HORAS,CANCELADO,2026-05-13,2025-11-11,2025-11-17,2025-11-17,2026-01-22,SEGTRUCK
2101,FRF4348,93ZM2SSH0E8826134,60020,60020,<NA>,23984,140570,UNIDADE MARINGÁ,ICARO GOMES,ATIVO,...,<NA>,677632,REPARAÇÃO OU REPOSIÇÃO DO VEÍCULO,CANCELADO,2026-05-13,2025-11-11,2025-11-17,2025-11-17,2026-01-22,SEGTRUCK
2102,FRF4348,93ZM2SSH0E8826134,60020,60020,<NA>,23984,140570,UNIDADE MARINGÁ,ICARO GOMES,ATIVO,...,<NA>,677631,REPARAÇÃO A TERCEIROS,CANCELADO,2026-05-13,2025-11-11,2025-11-17,2025-11-17,2026-01-22,SEGTRUCK


In [8]:
# DEFININDO TODAY TS e YESTERDAY

today_ts = pd.Timestamp.today().normalize()

if today_ts.weekday() == 0:  
    yesterday_ts = today_ts - dt.timedelta(days=3)
else:
    yesterday_ts = today_ts - dt.timedelta(days=1)
yesterday = yesterday_ts.strftime('%d-%m-%Y')

yesterday

'12-05-2026'

### JUNÇÃO DE BASES DE CANCELAMENTO

In [9]:
# ATUALIZANDO NOME COLUNA DE ATUALIZAÇÃO DE CANCELAMENTOS PARCIAIS

df_cancelamentos_parciais = df_cancelamentos_parciais.rename(columns={'data_atualizacao': 'data_cancelamento'})

In [10]:
# CRIANDO COLUNAS DE IDENTIFICADOR DE CANCELAMENTO

df_cancelamentos_integrais['identificador'] = 'INTEGRAL'
df_cancelamentos_parciais['identificador'] = 'PARCIAL'

In [11]:
# RETIRANDO DE CANCELAMENTOS PARCIAIS E DE CANCELAMENTOS INTEGRAIS AS PLACAS ATIVAS (PODEM TER SIDO RENOVADAS EM OUTRO CONJUNTO)

lista_ativos = df_ativos['chassi'].to_list()
df_cancelamentos_parciais = df_cancelamentos_parciais[~df_cancelamentos_parciais['chassi'].isin(lista_ativos)]


In [12]:
# CONCATENANDO BASES DE CANCELAMENTO E RETIRANDO DUPLICATAS POR CHASSI

df_cancelamentos = pd.concat(
    [df_cancelamentos_integrais, df_cancelamentos_parciais], ignore_index=True
)

# Converter data_cancelamento para Timestamp antes de sort_values para evitar erro de comparação
df_cancelamentos['data_cancelamento'] = pd.to_datetime(
    df_cancelamentos['data_cancelamento'], errors='coerce'
)

df_cancelamentos = df_cancelamentos.sort_values(
    by='data_cancelamento', ascending=False
).reset_index(drop=True)

df_cancelamentos = df_cancelamentos.drop_duplicates(subset=['chassi'], keep='first')

In [13]:
# ACRESCENTANDO COLUNA DE FRANQUEAMENTO OU NÃO

def map_franqueado(tem_palavra_chave):
    # Se for NA (pandas.NA), trata como "SIM"
    if pd.isna(tem_palavra_chave):
        return "SIM"
    return "UNIDADES/FRANQUIAS" if tem_palavra_chave else "PARCEIROS/CORRETORES/FTR"

df_cancelamentos['categoria_comercial'] = df_cancelamentos['unidade'].str.contains(
    "UNIDADE|MF|FRANQUEADO|MICRO|TS CONSULTORIA|TRANSDESK DIGITAL", na=False, case=False
).apply(map_franqueado)

In [14]:
# FORMATANDO COLUNAS DE DATA DE CANCELAMENTO PARA DT (TS)

df_cancelamentos['data_cancelamento'] = pd.to_datetime(
	df_cancelamentos['data_cancelamento'], errors='coerce'
).dt.date

In [15]:
# DEFININDO TODAY e YESTERDAY

today = pd.Timestamp.today().date()

yesterday = today - dt.timedelta(days=1)
sexta = yesterday - dt.timedelta(days=2)
sabado = yesterday - dt.timedelta(days=1)

### ACRESCENTANDO VISUALIZAÇÃO DE INADIMPLÊNCIA

In [16]:
#CRIANDO DATAFRAME DE INADIMPLÊNCIA 45+
caminho_query = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_inadimplencia\sql\faturas_45+.sql"
with open(caminho_query, 'r') as arquivo_query:
    query = arquivo_query.read()
df_faturas = awr.athena.read_sql_query(query, database='silver')

boletos_pagos = df_faturas.loc[df_faturas['historico'] != 1, 'ponteiro'].tolist()
df_inadimp = df_faturas[~df_faturas['ponteiro'].isin(boletos_pagos)]
df_inadimp = df_inadimp[df_inadimp['historico'] == 1]
df_inadimp = df_inadimp.drop_duplicates(subset=['ponteiro', 'conjunto', 'matricula', 'empresa'], keep='first')
# a pedido do setor de COBRANÇAS, retirar associado: APROSSIL - ASSOCIACAO DE PROPRIETARIOS DE CAMINHOES DO SUL D
# a pedido do setor de COBRANÇAS, retirar associado: TESTE
df_inadimp_atual_45 = df_inadimp[
    ~(
        (df_inadimp['empresa'].isin(['Viavante', 'Stcoop', 'Segtruck'])) &
        (df_inadimp['associado'] == "APROSSIL - ASSOCIACAO DE PROPRIETARIOS DE CAMINHOES DO SUL D")
    )
]
df_inadimp_atual_45 = df_inadimp_atual_45[~df_inadimp_atual_45['associado'].str.contains('TESTE', na=False)]

#caminho_pasta = r'C:\Users\raphael.almeida\OneDrive - Grupo Unus\analise de dados - Arquivos em excel\Relatório de Inadimplência'
#caminho_arquivo = os.path.join(caminho_pasta, 'relatorio_inadimplencia.xlsx')
#os.makedirs(caminho_pasta, exist_ok=True)
#if os.path.exists(caminho_arquivo):
#    os.remove(caminho_arquivo)
#    print("Arquivo antigo removido, iniciando carregamento...")
#df_inadimp_atual_45.to_excel(caminho_arquivo, engine='openpyxl', index=False, sheet_name='inadimplentes')
#print(f"Arquivo Excel salvo com sucesso em: {caminho_arquivo}")

In [17]:
# CONCATENANDO COM A BASE DE CANCELAMENTOS

df_cancelamentos['inadimplente_45+'] = df_cancelamentos['conjunto'].isin(df_inadimp_atual_45['conjunto']).map({True: 'SIM', False: 'NÃO'})

### CONSTRUINDO EXCEL

In [18]:
# DEFININDO CÉLULAS RELATÓRIO w4
df_cancelamentos['data_cancelamento'] = pd.to_datetime(df_cancelamentos['data_cancelamento'], errors='coerce').dt.date

if today.weekday() == 0: 
    ativos_mask = (df_ativos['inicio_vig'] >= sexta) & (df_ativos['inicio_vig'] <= yesterday)
    cancelamentos_mask = (df_cancelamentos['data_cancelamento'] >= sexta) & (df_cancelamentos['data_cancelamento'] <= yesterday)
else:
    ativos_mask = (df_ativos['inicio_vig'] == yesterday)
    cancelamentos_mask = (df_cancelamentos['data_cancelamento'] == yesterday)

ativados_seg = len(df_ativos[(df_ativos['empresa'] == 'SEGTRUCK') & ativos_mask])
ativados_st = len(df_ativos[(df_ativos['empresa'] == 'STCOOP') & ativos_mask])
ativados_viav = len(df_ativos[(df_ativos['empresa'] == 'VIAVANTE') & ativos_mask])
ativados_tag = len(df_ativos[(df_ativos['empresa'] == 'TAG') & ativos_mask])
total_ativados = len(df_ativos[ativos_mask])


cancelados_seg = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'SEGTRUCK') & cancelamentos_mask])
cancelados_st = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'STCOOP') & cancelamentos_mask])
cancelados_viav = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'VIAVANTE') & cancelamentos_mask])
cancelados_tag = len(df_cancelamentos[(df_cancelamentos['empresa'] == 'TAG') & cancelamentos_mask])
total_cancelados = len(df_cancelamentos[cancelamentos_mask])

In [19]:
# FUNÇÃO PARA LIMPAR DADOS DA PLANILHA
def clear_sheet(sheet):
    max_row = sheet.max_row
    if max_row > 1:
        sheet.delete_rows(1,max_row)

In [20]:
# DEFININDO NOMES DAS ABAS E FAZENDO A LIMPEZA DE w1 e w2

template = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\template\TEMPLATE_BASE_ATIVACOES_CANCELAMENTOS.xlsx"

wb = openpyxl.load_workbook(template)

w1 = wb['ATIVOS']
w2 = wb['CANCELAMENTOS']
w3 = wb['BASE']
w4 = wb['RELATORIO']

clear_sheet(w1)
clear_sheet(w2)

In [21]:
# FORMATANDO LINHAS NULAS

numeric_cols = df_ativos.select_dtypes(include=['number']).columns
string_cols = df_ativos.select_dtypes(include=['object', 'string']).columns

df_ativos[numeric_cols] = df_ativos[numeric_cols].fillna(0)
df_ativos[string_cols] = df_ativos[string_cols].fillna('N/A')

numeric_cols = df_cancelamentos.select_dtypes(include=['number']).columns
string_cols = df_cancelamentos.select_dtypes(include=['object', 'string']).columns

df_cancelamentos[numeric_cols] = df_cancelamentos[numeric_cols].fillna(0)
df_cancelamentos[string_cols] = df_cancelamentos[string_cols].fillna('N/A')

W1 - ATIVOS

In [22]:

for c_idx, col_name in enumerate(df_ativos.columns, start=1):
    w1.cell(row=1, column=c_idx, value=col_name)

if not df_ativos.empty:
    for r_idx, row in enumerate(df_ativos.values, start=2):
        for c_idx, value in enumerate(row, start=1):
            w1.cell(row=r_idx, column=c_idx, value=value)

W2 - CANCELADOS

In [23]:

for c_idx, col_name in enumerate(df_cancelamentos.columns, start=1):
    w2.cell(row=1, column=c_idx, value=col_name)

if  df_cancelamentos.empty == False:
    for r_idx, row in enumerate(df_cancelamentos.values, start=2):
        for c_idx, value in enumerate(row, start=1):
            w2.cell(row=r_idx, column=c_idx, value=value)

W3 - BASE

In [24]:
first_empty_row = 1
for row in range(1, w3.max_row + 1):
    if w3.cell(row=row, column=2).value is None:
        first_empty_row = row
        break
else:
    first_empty_row = w3.max_row + 1

# PEGAR DATA ATUAL NA PLANILHA
data_atual_planilha = w3['A' + str(first_empty_row)].value.date()


In [25]:
# VERIFICA SE DATA NA PLANILHA = DATA DE ONTEM  
#if data_atual_planilha == yesterday:
    # Filtra ativos até a data de ontem (inclusive)
if "inicio_vig" in df_ativos.columns:
    # Supondo que data_ativacao já seja datetime ou string convertível
    data_ativacao_col = pd.to_datetime(df_ativos["inicio_vig"], errors='coerce')
    df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(yesterday)]
    qtd_ativos = df_ativos_filtrados['chassi'].nunique()
else:
    qtd_ativos = df_ativos['chassi'].nunique()  # fallback: considera todos

#definindo ativados totais

data_ativacao_col = pd.to_datetime(df_ativos["inicio_vig"], errors='coerce')

df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(yesterday)]
qtd_ativos_dom = df_ativos_filtrados['chassi'].nunique()

df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(sabado)]
qtd_ativos_sab = df_ativos_filtrados['chassi'].nunique()

df_ativos_filtrados = df_ativos[data_ativacao_col <= pd.to_datetime(sexta)]
qtd_ativos_sex = df_ativos_filtrados['chassi'].nunique()


#definindo ativações de sexta e sábado
ativos_mask_dom = df_ativos['inicio_vig'] == yesterday
total_ativados_dom = len(df_ativos[ativos_mask_dom])

ativos_mask_sab = df_ativos['inicio_vig'] == sabado
total_ativados_sab = len(df_ativos[ativos_mask_sab])

ativos_mask_sex = df_ativos['inicio_vig'] == sexta
total_ativados_sex= len(df_ativos[ativos_mask_sex])

#definindo cancelamentos de sexta e sábado
cancel_mask_dom = df_cancelamentos['data_cancelamento'] == yesterday
total_cancel_dom = len(df_cancelamentos[cancel_mask_dom])

cancel_mask_sab = df_cancelamentos['data_cancelamento'] == sabado
total_cancel_sab = len(df_cancelamentos[cancel_mask_sab])

cancel_mask_sex = df_cancelamentos['data_cancelamento'] == sexta
total_cancel_sex = len(df_cancelamentos[cancel_mask_sex])

#hoje = dt.date.today()
if today.weekday() == 0 and first_empty_row >= 3:
        w3['B' + str(first_empty_row + 2)] = qtd_ativos_dom
        w3['C' + str(first_empty_row + 2)] = total_ativados_dom
        w3['D' + str(first_empty_row + 2)] = total_cancel_dom

        w3['B' + str(first_empty_row + 1)] = qtd_ativos_sab
        w3['C' + str(first_empty_row + 1)] = total_ativados_sab
        w3['D' + str(first_empty_row + 1)] = total_cancel_sab

        w3['B' + str(first_empty_row)] = qtd_ativos_sex
        w3['C' + str(first_empty_row)] = total_ativados_sex
        w3['D' + str(first_empty_row)] = total_cancel_sex

        print('Registro de ativos preenchido para domingo, sábado e ativos na aba BASE!')
else:
        w3['B' + str(first_empty_row)] = qtd_ativos
        w3['C' + str(first_empty_row)] = total_ativados
        w3['D' + str(first_empty_row)] = total_cancelados
        



W4 - RESUMO

In [ ]:
import datetime 
# Garante que yesterday esteja normalizado
if isinstance(yesterday, pd.Timestamp):
    yest_date = yesterday.date()
elif isinstance(yesterday, datetime.datetime):
    yest_date = yesterday.date()
elif isinstance(yesterday, datetime.date):
    yest_date = yesterday
else:
    yest_date = pd.to_datetime(yesterday).date()

# Hoje
hoje = datetime.date.today()
dia_semana = hoje.weekday() 

if dia_semana == 0:  # Segunda-feira
    # calcula sexta (3 dias atrás) e domingo (ontem)
    sexta = yest_date - datetime.timedelta(days=2)
    domingo = yest_date
    sexta_str = sexta.strftime('%d/%m/%Y')
    domingo_str = domingo.strftime('%d/%m/%Y')
    resumo_periodo = f"{sexta_str} (sexta) - {domingo_str} (domingo)"
    w4['C2'] = resumo_periodo
else:
    w4['C2'] = yest_date.strftime('%d/%m/%Y')

w4['C3'] = qtd_ativos

w4['C8'] = ativados_seg
w4['C9'] = ativados_st
w4['C10'] = ativados_viav
w4['C11'] = ativados_tag

w4['D6'] = cancelados_seg
w4['D7'] = cancelados_st
w4['D8'] = cancelados_viav
w4['D9'] = cancelados_tag


### automação envio whatsapp

In [27]:
# time.sleep(2)
# pyautogui.hotkey('win', 'e')
# time.sleep(4)
# pyautogui.hotkey('ctrl', 'l') 
# time.sleep(1.5)
# pyautogui.typewrite(r'C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\img')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'f') 
# time.sleep(2.5)
# pyautogui.typewrite(r'graficos')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(1.5)
# pyautogui.press('down')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'c')
# #whatsapp - primeiro arquivo
# time.sleep(1.5)
# pyautogui.press('win')
# time.sleep(1.5)
# pyautogui.typewrite('whatsapp')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(5)
# pyautogui.hotkey('ctrl', 'f') 
# time.sleep(1.5)
# pyautogui.typewrite("RELAT")
# time.sleep(1.5)
# pyautogui.press('down')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'v')
# time.sleep(2.5)
# pyautogui.press('enter')
# time.sleep(1.5)
######################################################whatsapp - segundo arquivo
# pyautogui.hotkey('alt', 'tab')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'f') 
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'a')
# time.sleep(2.5)
# pyautogui.typewrite(r'tabela')
# time.sleep(1.5)
# pyautogui.press('enter')
# time.sleep(1.5)
# pyautogui.press('down')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'c')
# time.sleep(1.5)
# pyautogui.hotkey('alt', 'tab')
# time.sleep(1.5)
# pyautogui.hotkey('ctrl', 'v')
# time.sleep(2.5)
# pyautogui.press('enter')


### carregando resultados em excel

In [28]:
import os

# save workbook to the full file path
wb.save(template)

name_file = fr'RELATORIO_ATIVACOES_CANCELAMENTOS_{yesterday}.xlsx'

output_path = r"C:\Users\raphael.almeida\Documents\Processos\relatorio_ativacoes_cancelamentos\reports"
os.makedirs(output_path, exist_ok=True)
output_full_path = os.path.join(output_path, name_file)

# save workbook to the full file path
shutil.copy(template, output_full_path)

name_file_2 = fr'RELATORIO_ATIVACOES_CANCELAMENTOS.xlsx'

path_sharepoint = r"C:\Users\raphael.almeida\OneDrive - Grupo Unus\analise de dados - Arquivos em excel"
full_path_sharepoint = os.path.join(path_sharepoint, name_file_2)

# copy the saved file to the SharePoint folder (src, dst)
shutil.copy(output_full_path, full_path_sharepoint)

wb.close()